In [1]:
import numpy as np
import pandas as pd
import xarray as xr

In [2]:
from data import UncertainDataset, ds_single, ds_pressure

In [3]:
from pathlib import Path

plots = Path("plots")

In [4]:
from matplotlib import pyplot as plt

def plot_ensemble_mean_spread(mean: xr.Dataset, spread: xr.Dataset, plot: Path, sample: slice = slice(None)):
    fig, ax = plt.subplots()

    m0, m1, m99, m100 = np.nanquantile(mean.values.flatten(), [0.0, 0.01, 0.99, 1.0])
    s99, s100 = np.nanquantile(spread.values.flatten(), [0.99, 1.0])

    months = sorted(set(zip(
        pd.to_datetime(mean.valid_time).year,
        pd.to_datetime(mean.valid_time).month,
    )))
    colours = ["tab:orange", "tab:blue"]
    
    ax.set_title(mean.long_name)
    ax.set_xlabel(f"mean [{mean.units}]")
    ax.set_ylabel(f"spread [{spread.units}]")

    for (year, month), colour in zip(months, colours):
        ax.scatter(
            mean.sel(valid_time=f"{year}-{month:02}").values.flatten()[sample],
            spread.sel(valid_time=f"{year}-{month:02}").values.flatten()[sample],
            s=1, label=f"{year}/{month:02}", edgecolor="none", marker=".", c=colour,
        )

    if m1 != m99:
        ms = m99 - m1
        ax.set_xlim(m1 - ms*0.05, m99 + ms*0.05)
    else:
        ms = m100 - m0
        ax.set_xlim(m0 - ms*0.05, m100 + ms*0.05)
    if s99 != 0.0:
        ss = s99
        ax.set_ylim(0.0 - ss*0.05, s99 + ss*0.05)
    else:
        ss = s99
        ax.set_ylim(0.0 - ss*0.05, s100 + ss*0.05)

    fig.legend(loc="upper right", markerscale=10)

    plot.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(plot)

    plt.close(fig)

In [5]:
from tqdm import tqdm

In [ ]:
for key in tqdm(sorted(ds_single.mean)):
    plot_ensemble_mean_spread(ds_single.mean[key], ds_single.spread[key], plots / "single" / f"{key}.png", slice(0, None, 1000))

  0%|          | 0/226 [00:00<?, ?it/s]/tmp/junityre/26911954/ipykernel_1943299/578804521.py:37: UserWarning: Attempting to set identical bottom == top == 0.0 results in singular transformations; automatically expanding.
  ax.set_ylim(0.0 - ss*0.05, s100 + ss*0.05)
  0%|          | 1/226 [00:11<42:55, 11.45s/it]/tmp/junityre/26911954/ipykernel_1943299/578804521.py:37: UserWarning: Attempting to set identical bottom == top == 0.0 results in singular transformations; automatically expanding.
  ax.set_ylim(0.0 - ss*0.05, s100 + ss*0.05)
  1%|          | 2/226 [00:23<44:16, 11.86s/it]/tmp/junityre/26911954/ipykernel_1943299/578804521.py:37: UserWarning: Attempting to set identical bottom == top == 0.0 results in singular transformations; automatically expanding.
  ax.set_ylim(0.0 - ss*0.05, s100 + ss*0.05)
  1%|▏         | 3/226 [00:34<42:08, 11.34s/it]/tmp/junityre/26911954/ipykernel_1943299/578804521.py:37: UserWarning: Attempting to set identical bottom == top == 0.0 results in singular

In [ ]:
for key in tqdm(sorted(ds_pressure.mean)):
    for pressure in ds_pressure.mean[key].pressure_level.values:
        plot_ensemble_mean_spread(
            ds_pressure.mean[key].sel(pressure_level=pressure),
            ds_pressure.spread[key].sel(pressure_level=pressure),
            plots / "pressure" / f"{key}-{pressure}hPa.png",
            slice(0, None, 1000),
        )